In [30]:
!pip install -q litellm langchain langchain-community langchain-openai python-dotenv redis langchain-litellm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai 0.108.0 requires litellm==1.60.2, but you have litellm 1.80.0 which is incompatible.
pyopenssl 24.2.1 requires cryptography<44,>=41.0.5, but you have cryptography 46.0.7 which is incompatible.


In [18]:
import warnings
import logging 


warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)


from litellm import completion
import litellm
litellm.suppress_debug_info = True 


In [35]:
import os
import ssl
import httpx
from dotenv import load_dotenv

load_dotenv()


ssl._create_default_https_context = ssl._create_unverified_context

# Also set standard environment variables to bypass SSL verification for requests/httpx if needed
os.environ["CURL_CA_BUNDLE"] = ""
os.environ["REQUESTS_CA_BUNDLE"] = ""



In [36]:
from litellm import completion
import httpx
import openai
import os

# Create an OpenAI client with an unverified HTTP client to bypass local network SSL inspection
custom_client = openai.OpenAI(http_client=httpx.Client(verify=False))

response_openai=completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Explain alignment in LLMs in simple terms."}],
client=custom_client
)

print("OpenAI Response:",response_openai.choices[0].message.content)


custom_groq_client = openai.OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
    http_client=httpx.Client(verify=False)
)

response_groq=completion(
    model="groq/llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Explain alignment in LLMs in simple terms."}],
client=custom_groq_client
)

print("GROQ Response:",response_groq.choices[0].message.content)

OpenAI Response: Alignment in large language models (LLMs) refers to the process of ensuring that the model's behavior and responses are consistent with human values, intentions, and ethical standards. Essentially, it means making sure that when the model generates text, it aligns with what people consider appropriate, helpful, and safe.

Here’s a simple breakdown:

1. **Understanding Human Intent**: The model should interpret what users want and respond accordingly. For example, if someone asks for advice, the model should provide something sensible and constructive.

2. **Ethical Standards**: The model should avoid generating harmful or misleading content. This includes steering clear of hate speech, misinformation, or any other type of dangerous content.

3. **User Safety**: The model should prioritize the safety and well-being of users. If a question could lead to harmful behavior or decisions, the model should know how to handle it appropriately.

4. **Cultural Sensitivity**: Diff

In [15]:
from litellm import completion
import httpx
import openai
import os

custom_http_client = httpx.Client(verify=False)

prompt = "Explain alignment in LLMs in simple terms."

providers=[
# Note that client allows bypassing SSL verify
    ("OpenAI","gpt-4o-mini", {"client": openai.OpenAI(http_client=custom_http_client)}),
    ("GROQ","groq/llama-3.3-70b-versatile", {"client": openai.OpenAI(api_key=os.environ.get("GROQ_API_KEY"), base_url="https://api.groq.com/openai/v1", http_client=custom_http_client)}),
("Anthropic","claude-3-5-haiku-20241022", {"client": openai.OpenAI(api_key=os.environ.get("ANTHROPIC_API_KEY"), base_url="https://api.anthropic.com/v1", http_client=custom_http_client)}),
    ("Gemini","gemini/gemini-1.5-flash", {"client": openai.OpenAI(api_key=os.environ.get("GOOGLE_API_KEY"), base_url="https://generativelanguage.googleapis.com/v1beta/openai", http_client=custom_http_client)}),
]

for label,model,kwargs in providers:
    try:
        r=completion(
            model=model,
            messages=[
                {"role": "user", "content": prompt}
            ],
            **kwargs
        )
        print(f"{label} Response:", r.choices[0].message.content)
    
    except Exception as e:
        print(f"{label}:{type(e).__name__} - {e}")

OpenAI Response: Alignment in large language models (LLMs) refers to the process of ensuring that the model's behavior and responses are consistent with human values, intentions, and ethical standards. Essentially, it means making sure that when the model generates text, it aligns with what people consider appropriate, helpful, and safe.

Here’s a simple breakdown:

1. **Understanding Human Intent**: The model should interpret what users want and respond accordingly. For example, if someone asks for advice, the model should provide something sensible and constructive.

2. **Ethical Standards**: The model should avoid generating harmful or misleading content. This includes steering clear of hate speech, misinformation, or any other type of dangerous content.

3. **User Safety**: The model should prioritize the safety and well-being of users. If a question could lead to harmful behavior or decisions, the model should know how to handle it appropriately.

4. **Cultural Sensitivity**: Diff

Fallbacks

In [19]:
from litellm import completion

response = completion(
    model="groq/llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Explain alignment in LLMs in simple terms."}],
    fallbacks=[
        {"model": "anthropic/claude-3-5-haiku-20241022"},
        {"model": "gemini/gemini-1.5-flash"},
        {"model": "gpt-4o-mini"}

    ])

print("Response :", response.choices[0].message.content)
print("Model used:", response.model)

Task was destroyed but it is pending!
task: <Task pending name='Task-8' coro=<Logging.async_success_handler() running at /opt/anaconda3/lib/python3.12/site-packages/litellm/litellm_core_utils/litellm_logging.py:1516>>


Response : Alignment in Large Language Models (LLMs) refers to the process of ensuring that the model's goals and behaviors are in line with human values and intentions. Think of it like teaching a child to behave well in society.

Imagine you have a highly intelligent and capable child who can learn and adapt quickly. However, if you don't teach them what's right and wrong, they might use their abilities for bad things. Similarly, LLMs are highly capable models that can learn and generate human-like text, but if they're not aligned with human values, they might produce harmful or undesirable content.

Alignment involves training and fine-tuning the model to:

1. **Understand human values**: Teach the model what's important to humans, such as kindness, respect, and fairness.
2. **Avoid harm**: Prevent the model from generating content that's harmful, offensive, or biased.
3. **Follow guidelines**: Ensure the model follows rules and guidelines set by humans, such as laws and regulations

Cost Tracking

In [33]:
from litellm import completion,completion_cost

custom_client = openai.OpenAI(http_client=httpx.Client(verify=False))

response_openai=completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Explain alignment in LLMs in simple terms."}],
    client=custom_client
)

cost=completion_cost(completion_response=response)

print("Response:", response.choices[0].message.content)
print("Input tokens:",response.usage.prompt_tokens)
print("Output tokens:",response.usage.completion_tokens)
print("Cost:", cost)

21:34:25 - LiteLLM:ERROR: caching.py:629 - LiteLLM Cache: Excepton add_cache: __annotations__
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/site-packages/litellm/caching/caching.py", line 624, in add_cache
    cache_key, cached_data, kwargs = self._add_cache_logic(
                                     ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/litellm/caching/caching.py", line 608, in _add_cache_logic
    raise e
  File "/opt/anaconda3/lib/python3.12/site-packages/litellm/caching/caching.py", line 588, in _add_cache_logic
    cache_key = self.get_cache_key(**kwargs)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/litellm/caching/caching.py", line 250, in get_cache_key
    combined_kwargs = self._get_relevant_args_to_use_for_cache_key()
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/litellm/caching/caching.p

Response: Alignment in Large Language Models (LLMs) refers to the process of ensuring that the model's goals and behaviors are in line with human values and intentions. Think of it like teaching a child to behave well in society.

Imagine you have a highly intelligent and capable child who can learn and adapt quickly. However, if you don't teach them what's right and wrong, they might use their abilities for bad things. Similarly, LLMs are highly capable models that can learn and generate human-like text, but if they're not aligned with human values, they might produce harmful or undesirable content.

Alignment involves training and fine-tuning the model to:

1. **Understand human values**: Teach the model what's important to humans, such as kindness, respect, and fairness.
2. **Avoid harm**: Prevent the model from generating content that's harmful, offensive, or biased.
3. **Follow guidelines**: Ensure the model follows rules and guidelines set by humans, such as laws and regulations.

Caching

In [22]:
import litellm

litellm.callbacks=[]
litellm.success_callback=[]
litellm.failure_callback=[]
litellm._async_success_callback=[]
litellm._async_failure_callback=[]

litellm.cache=None

print("Litellm state reset-ready for clean caching demo")

Litellm state reset-ready for clean caching demo


In [9]:
import litellm
import time
from litellm import completion
from litellm.caching import Cache


litellm.cache = Cache(type="redis", host="localhost", port=6379)

prompt="What are LLM gateways used for?"

start=time.time()

r1=completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    client=custom_client
)

end=time.time()
t1=end-start
print("First call response:",r1.choices[0].message.content)
print("Time taken for first call:", t1)


start1=time.time()

r2=completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    client=custom_client
)
end1=time.time()
t2=end1-start1
print("Second call response:",r2.choices[0].message.content)
print("Time taken for second call:", t2)

print("Time difference between first and second call (cache hit):", t1-t2)
print("Cache hit ratio:", t1/t2)





First call response: LLM gateways, or Large Language Model gateways, are used for a variety of purposes in the field of artificial intelligence and natural language processing. Here are some common use cases and functionalities of LLM gateways:

1. **Access and Integration**: LLM gateways facilitate the integration of large language models into applications, allowing developers to easily access the capabilities of models without needing to manage the underlying infrastructure.

2. **API Management**: They often provide an API (Application Programming Interface) through which applications can communicate with the language model, enabling features like text generation, summarization, translation, and conversation.

3. **Customization and Fine-Tuning**: Some LLM gateways offer features for customizing or fine-tuning the language model on specific datasets or for particular applications, allowing users to tailor the model’s responses to their needs.

4. **Scalability**: LLM gateways are ty

Router


In [25]:
import os 
import httpx
import openai
from litellm import Router

custom_http_client = httpx.Client(verify=False)


groq_client_unverified = openai.OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"), 
    base_url="https://api.groq.com/openai/v1", 
    http_client=custom_http_client
)
openai_client_unverified = openai.OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
    http_client=custom_http_client
)

model_list=[
    {
        "model_name":"fast-cheap",
        "litellm_params": {
            "model":"groq/llama-3.3-70b-versatile",
            "api_key": os.environ.get("GROQ_API_KEY"),
            "api_base": "https://api.groq.com/openai/v1"
        }
    },
    {
        "model_name":"smart-coding",
        "litellm_params": {
            "model":"gpt-4o-mini",
            "api_key": os.environ.get("OPENAI_API_KEY"),
        }
    },
    {
        "model_name":"balanced",
        "litellm_params": {
            "model":"gpt-4o-mini",
            "api_key": os.environ.get("OPENAI_API_KEY"),
        }
    }
]

router = Router(model_list=model_list, routing_strategy="random")


prompt_simple = "What is the capital of France?"
prompt_coding = "Write a python function to show LLM alignment in simple terms."


print("--- Routing simple prompt to 'fast-cheap' model ---")
response_fast = router.completion(
    model="fast-cheap",
    messages=[{"role": "user", "content": prompt_simple}],
    client=groq_client_unverified
)
print("Response:\n", response_fast.choices[0].message.content)

print("\n--- Routing coding prompt to 'smart-coding' model ---")
response_coding = router.completion(
    model="smart-coding",
    messages=[{"role": "user", "content": prompt_coding}],
    client=openai_client_unverified
)
print("Response:\n", response_coding.choices[0].message.content)




Intialized router with Routing strategy: random

Routing fallbacks: None

Routing context window fallbacks: None

Router Redis Caching=None
--- Routing simple prompt to 'fast-cheap' model ---
Response:
 The capital of France is Paris.

--- Routing coding prompt to 'smart-coding' model ---
Response:
 The capital of France is Paris.

--- Routing coding prompt to 'smart-coding' model ---
Response:
 Sure! To explain the concept of LLM (Large Language Model) alignment in simple terms, we can create a Python function that simulates a conversation with an LLM. The function will demonstrate how to align the model's responses with the user's intentions or values.

Here's a simple example of a Python function that reflects this concept:

```python
def llm_alignment(user_input):
    # Simple alignment rules
    alignment_rules = {
        "thank you": "You're welcome! I'm here to help you.",
        "help": "Of course! What do you need help with?",
        "joke": "Why did the scarecrow win an aw

Load Balancing

In [29]:
import os 
import httpx
import openai
from litellm import Router

custom_http_client = httpx.Client(verify=False)


groq_client_unverified = openai.OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"), 
    base_url="https://api.groq.com/openai/v1", 
    http_client=custom_http_client
)
openai_client_unverified = openai.OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
    http_client=custom_http_client
)

model_list=[
    {
        "model_name":"fast-cheap",
        "litellm_params": {
            "model":"groq/llama-3.3-70b-versatile",
            "api_key": os.environ.get("GROQ_API_KEY"),
            "api_base": "https://api.groq.com/openai/v1"
        }
    },
    {
        "model_name":"smart-coding",
        "litellm_params": {
            "model":"gpt-4o-mini",
            "api_key": os.environ.get("OPENAI_API_KEY"),
        }
    },
    {
        "model_name":"balanced",
        "litellm_params": {
            "model":"gpt-4o-mini",
            "api_key": os.environ.get("OPENAI_API_KEY"),
        }
    }
]

router = Router(model_list=model_list, routing_strategy="simple-shuffle")

prompt_simple = "What is the capital of France?"

print(f"{'Request':<10}{'Deployment Picked':<22}{'Latency (s)':<15}{'Response':<50}")
print("-"*100)


for i in range(6):
    response = router.completion(
        model="balanced",
        messages=[{"role": "user", "content": prompt_simple}],
        
        client=openai_client_unverified
    )
   
    model_picked = getattr(response, "model", "Unknown")

    
    latency = getattr(response, "_hidden_params", {}).get("model_response_time", 0.0)
    
    print(f"{i+1:<10}{model_picked:<22}{latency:<15.2f}{response.choices[0].message.content[:50]:<50}")


Intialized router with Routing strategy: simple-shuffle

Routing fallbacks: None

Routing context window fallbacks: None

Router Redis Caching=None
Request   Deployment Picked     Latency (s)    Response                                          
----------------------------------------------------------------------------------------------------
1         gpt-4o-mini-2024-07-180.00           The capital of France is Paris.                   
1         gpt-4o-mini-2024-07-180.00           The capital of France is Paris.                   
2         gpt-4o-mini-2024-07-180.00           The capital of France is Paris.                   
2         gpt-4o-mini-2024-07-180.00           The capital of France is Paris.                   
3         gpt-4o-mini-2024-07-180.00           The capital of France is Paris.                   
3         gpt-4o-mini-2024-07-180.00           The capital of France is Paris.                   
4         gpt-4o-mini-2024-07-180.00           The capital of Fra

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import httpx
import os

# Note: The native LangChain litellm wrapper can struggle copying custom OpenAI unverified clients across invocations.
# Instead, using LangChain's native ChatOpenAI class safely accepts an `http_client` bypass property.

unverified_client = httpx.Client(verify=False)

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.environ.get("OPENAI_API_KEY"),
    http_client=unverified_client
)

prompt=ChatPromptTemplate.from_messages([
    ("system", "You are a helpful tutor. Be concise."),
    ("user", "{question}")
])

chain=prompt | llm | StrOutputParser()

answer=chain.invoke({"question":"Explain LLM alignment in simple terms."})
print("Answer:", answer)

Answer: LLM alignment refers to making sure that large language models (like AI systems) understand and act according to human values and intentions. The goal is to ensure these models produce responses that are safe, relevant, and beneficial to users, rather than harmful or misleading. This involves training the models with careful guidelines and feedback to ensure they behave in ways that align with what humans consider good and appropriate.


In [42]:
from langchain_openai import ChatOpenAI
from langchain_community.chat_models import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import httpx
import os


primary=ChatLiteLLM(
    model="gpt-o1", 
    api_key=os.environ.get("OPENAI_API_KEY"),
    api_base="https://api.openai.com/v1",
)

fallback_1=ChatLiteLLM(
    model="gpt-5", # another likely failed model
    api_key=os.environ.get("OPENAI_API_KEY"),
    api_base="https://api.openai.com/v1",
)

fallback_2=ChatLiteLLM(
    model="groq/llama-3.3-70b-versatile",
    api_key=os.environ.get("GROQ_API_KEY"),
    api_base="https://api.groq.com/openai/v1",
)


robust_llm=primary.with_fallbacks([fallback_1, fallback_2])

prompt=ChatPromptTemplate.from_messages([
    ("system", "You are an AI engineer, say you are in an interview. Be concise."),
    ("user", "{question}")
])


chain=prompt | robust_llm | StrOutputParser()

result=chain.invoke({"question":"What are the tradeoffs in general RAG systems"})
print("Final Result:\n", result)

Retrying langchain_community.chat_models.litellm.ChatLiteLLM.completion_with_retry.<locals>._completion_with_retry in 4.0 seconds as it raised APIConnectionError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=gpt-o1
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers.
Retrying langchain_community.chat_models.litellm.ChatLiteLLM.completion_with_retry.<locals>._completion_with_retry in 4.0 seconds as it raised APIConnectionError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=gpt-o1
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers.
Retrying langchain_community.chat_models.litellm.ChatLiteLLM.completion_with_retry.<locals>._completion_with_retry in 4.0 seconds as it raised A

Final Result:
 In general RAG (Retrieval-Augmented Generation) systems, tradeoffs include:

1. **Accuracy vs. Efficiency**: More retrieval steps can improve accuracy but increase computational cost and latency.
2. **Knowledge Coverage vs. Noise**: Larger knowledge bases can provide more comprehensive coverage but also introduce noise and irrelevant information.
3. **Generative Power vs. Retrieval Reliance**: Stronger generative models can reduce reliance on retrieval, but may also lead to hallucinations or inaccuracies.
4. **Diversity vs. Coherence**: Encouraging diversity in generated responses can lead to loss of coherence and context understanding.

These tradeoffs require careful system design and tuning to balance competing objectives.
